# 1 таск

In [ ]:
# !pip install -q --upgrade transformers==4.57.1 trl==0.24.0 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.47.0 datasets==3.6.0

In [1]:
import random
from pathlib import Path

import json

import numpy as np
import torch

SEED = 42
DATA_DIR = Path("/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
def load_jsonl(name):
    path = DATA_DIR / "data" / name
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file]

kid_adult = load_jsonl("kid_adult.jsonl")
good_bad = load_jsonl("good_bad.jsonl")
test_style = load_jsonl("public_test_style.jsonl")
test_quality = load_jsonl("public_test_quality.jsonl")

In [3]:
kid_adult[1]

{'prompt': 'Что такое химическая реакция? Это процесс, при котором одни вещества превращаются в совершенно другие.',
 'kid': 'Это когда вещества встречаются и превращаются во что-то совсем новое, как когда из сырых продуктов в духовке получается вкусный пирог. Настоящее волшебство происходит, когда обычное тесто меняется и становится румяным хлебом. Старые вещества исчезают, а на их месте появляются совершенно другие.',
 'adult': 'Химическая реакция — это процесс превращения исходных веществ (реагентов) в новые соединения (продукты) путем разрыва и образования химических связей между атомами. При этом состав атомов в системе сохраняется, но их пространственное строение и свойства кардинально меняются. Процесс всегда сопровождается поглощением или выделением энергии в виде тепла или света.'}

In [4]:
good_bad[0]

{'instruction': 'Как кнопка на клавиатуре печатает букву на экране? Ответ: При нажатии контакты внутри замыкаются, и сигнал передается в мозг компьютера, который рисует нужную букву.',
 'chosen': 'Под кнопкой есть маленькая кнопочка-переключатель. Когда ты её нажимаешь, она посылает компьютеру тайный код из электричества. Компьютер мгновенно разгадывает этот код и рисует нужную букву на экране, прямо как волшебник!',
 'rejected': 'Знаешь, под каждой кнопочкой на твоей клавиатуре спрятана специальная маленькая пружинка и крошечная площадка. Как только ты нажимаешь пальцем на кнопку, она опускается вниз, и эта площадочка соединяет две проводящие штучки вместе. В этот самый момент по проводочку внутри компьютера бежит невидимый электрический сигнал, который очень-очень быстро добегает до самого главного мозга, процессора. Процессор тут же понимает, какую именно букву ты хотел написать, и отправляет команду экрану, чтобы она тут же появилась!'}

In [5]:
test_style[0]

{'prompt': 'Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.',
 'kid': 'Магазину нужно освободить полки, чтобы поставить новые игрушки и вещи. Поэтому старые товары продают дешевле, как будто кричат: «Забирайте скорее!». Это здорово помогает быстро всё разобрать и порадовать покупателей.',
 'adult': 'Продавцы делают распродажи для ускорения оборачиваемости товаров и освобождения складских запасов перед закупкой новых коллекций. Маркетинговое снижение цены стимулирует покупательский спрос и позволяет конвертировать неликвидные активы в живые деньги, избегая замораживания капитала.',
 'fact': 'Скидки и распродажи используются для ускорения сбыта остатков, расчистки складов под новые поступления и стимуляции спроса.'}

In [6]:
test_quality[0]

{'prompt': 'Почему у металла и у дерева разная температура в одной комнате? Ответ: Металл очень быстро забирает тепло у нашей руки, поэтому нам он кажется холоднее дерева.',
 'chosen': 'На самом деле они одинаковые! Просто металл — отличный мостик для тепла, поэтому он очень быстро забирает его у нашей руки, и нам кажется, что он холоднее.',
 'rejected': 'Знаешь, это очень-очень интересный вопрос! Представь себе, что они оба совершенно одинаковой температуры, как и сам воздух в твоей комнате. Но дело в том, что когда ты дотрагиваешься до металла ручкой, он просто-напросто начинает быстро-быстро вытягивать всё тепло из твоей руки. Дерево тоже немножко забирает тепло, но оно делает это совсем-совсем медленно, поэтому наша рука не успевает замёрзнуть, и нам кажется, что оно более тёпленькое и приятное на ощупь!'}

In [7]:
# ! pip install -U bitsandbytes

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
 

In [10]:
from datasets import Dataset

examples = []

for row in kid_adult:
    example = {
        "prompt": [
            {"role": "user", "content": row["prompt"]}
        ],
        "completion": [
            {"role": "assistant", "content": row["kid"]}
        ],
    }

    examples.append(example)

sft_data = Dataset.from_list(examples)

In [11]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

In [12]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [13]:
# ! pip install trl

In [14]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="sft_style",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=512,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_data,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as s

Step,Training Loss
25,1.558200
50,1.326600
75,1.261700
100,1.274200
125,1.225700
150,1.210300
175,1.191400
200,1.044700
225,0.880800
250,0.846600


TrainOutput(global_step=561, training_loss=0.9069820504350034, metrics={'train_runtime': 3246.6426, 'train_samples_per_second': 1.376, 'train_steps_per_second': 0.173, 'total_flos': 1.33823652761088e+16, 'train_loss': 0.9069820504350034, 'entropy': 0.9636641542116801, 'num_tokens': 608310.0, 'mean_token_accuracy': 0.8478224564481665, 'epoch': 3.0})

In [16]:
adapter_path = "/kaggle/working/sft_style_adapter"

trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

('/kaggle/working/sft_style_adapter/tokenizer_config.json',
 '/kaggle/working/sft_style_adapter/special_tokens_map.json',
 '/kaggle/working/sft_style_adapter/chat_template.jinja',
 '/kaggle/working/sft_style_adapter/vocab.json',
 '/kaggle/working/sft_style_adapter/merges.txt',
 '/kaggle/working/sft_style_adapter/added_tokens.json',
 '/kaggle/working/sft_style_adapter/tokenizer.json')

In [17]:
import pickle

model.config.use_cache = True
model.eval()

with open(DATA_DIR / "metrics" / "style_clf.pkl", "rb") as file:
    style_clf = pickle.load(file)

answers = []

for row in test_style:
    messages = [
        {"role": "user", "content": row["prompt"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    start = inputs["input_ids"].shape[1]
    new_tokens = output[0][start:]

    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    answers.append(answer)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

AttributeError: 'dict' object has no attribute 'predict_proba'

In [18]:
style_clf.keys()

dict_keys(['clf', 'vecs'])

## P_simple подсчет

In [19]:
from scipy.sparse import hstack

vectors = []

for vec in style_clf["vecs"]:
    vector = vec.transform(answers)
    vectors.append(vector)

features = hstack(vectors)

probabilities = style_clf["clf"].predict_proba(features)[:, 1]
p_simple = float(np.mean(probabilities))

print("P_simple:", p_simple)

P_simple: 0.9703538977842824


# 2 таск